In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('Breast Cancer METABRIC.csv')
target_col = 'Tumor Stage'
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' does not exist!")
stage4_count = (df[target_col] == 4).sum()
df.loc[df[target_col] == 4, target_col] = 3
df = df.dropna(subset=[target_col])
leakage_features = [
    'Overall Survival (Months)',
    'Overall Survival Status',
    'Patient\'s Vital Status',
    'Relapse Free Status (Months)',
    'Relapse Free Status',
    'Nottingham prognostic index',
]

removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)

# Delete features with high missing rate
missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"{len(high_missing)} high missing rate features have been deleted")

# Extract candidate features
exclude_kw = ['stage', 'unnamed', 'patient id', 'patient\'s vital',
              'survival', 'relapse', 'status']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)

print(f"{len(feature_cols)} candidate features identified")

X = df[feature_cols].copy()
y = df[target_col].copy()

# Handle missing values
for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)

# Label encoding
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

print(f"Feature matrix: {X.shape}")

# Target variable encoding
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print(f"\nTarget class mapping:")
for i, label in enumerate(target_encoder.classes_):
    print(f"  Stage {label} → {i}")

y = pd.Series(y_encoded, index=y.index)

# Feature selection
K_FEATURES = 15
selector = SelectKBest(score_func=mutual_info_classif, k=min(K_FEATURES, len(feature_cols)))
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(feature_scores.head(20).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (Score: {score:.4f})")

X_selected = X[selected_features].copy()

# Data splitting
class_counts = y.value_counts()
print("\nSample count by stage:")
for stage, count in class_counts.sort_index().items():
    original_label = target_encoder.classes_[stage]
    print(f"  Stage {original_label}: {count} cases")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# SMOTE
min_class_samples = y_train.value_counts().min()
k_neighbors = min(5, min_class_samples - 1)

if k_neighbors >= 1:
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
else:
    print("Sample size too small for SMOTE, using original training set")
    X_train_bal, y_train_bal = X_train, y_train

print(f"\nBefore SMOTE: {X_train.shape}")
print(f"After SMOTE: {X_train_bal.shape}")
print(f"Added samples: {X_train_bal.shape[0] - X_train.shape[0]}")

print("\nTraining set distribution (after SMOTE):")
for s, c in pd.Series(y_train_bal).value_counts().sort_index().items():
    original_label = target_encoder.classes_[int(s)]
    print(f"  Stage {original_label}: {c} cases")

# RF
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    verbose=0
)
rf_model.fit(X_train_bal, y_train_bal)

# Cross-validation
cv_scores = cross_val_score(rf_model, X_selected, y, cv=5, scoring='accuracy')
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nMean: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# Importance of selected features
final_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nModel importance of these {len(selected_features)} features:")
print(final_importance.to_string(index=False))

# Model evaluation
y_train_pred = rf_model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall    = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1        = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)

y_test_pred = rf_model.predict(X_test)

test_accuracy  = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_recall    = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1        = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print("=" * 70)
print(" " * 20 + "Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy:>15.4f} {test_accuracy:>15.4f} {abs(train_accuracy-test_accuracy):>15.4f}")
print(f"{'Precision':<15} {train_precision:>15.4f} {test_precision:>15.4f} {abs(train_precision-test_precision):>15.4f}")
print(f"{'Recall':<15} {train_recall:>15.4f} {test_recall:>15.4f} {abs(train_recall-test_recall):>15.4f}")
print(f"{'F1 Score':<15} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1-test_f1):>15.4f}")
print("=" * 70)

target_names = [f"Stage {target_encoder.classes_[i]}" for i in sorted(y_test.unique())]
print(classification_report(y_test, y_test_pred, target_names=target_names, zero_division=0))